In [1]:
## importing the important libraries:
import os
from dotenv import load_dotenv
load_dotenv()

from azure.core.credentials import AzureKeyCredential
from azure.ai.documentintelligence.aio import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import DocumentAnalysisFeature
from azure.ai.documentintelligence.models import DocumentContentFormat
from openai import AsyncAzureOpenAI
import base64


In [2]:
file_path = r"C:\Users\SumeetMaheshwari\Desktop\Data\Alembic data\Doc comparator\Input 1_pdf.pdf"

# Load environment variables
endpoint = os.getenv("DOCUMENT_INTELLIGENCE_ENDPOINT")
key = os.getenv("DOCUMENT_INTELLIGENCE_KEY")

In [3]:
### function for the reading the pdf as input and return the result
document_intelligence_client = DocumentIntelligenceClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(key)
)

async def data_extractor(pdf_path:str):
    with open(pdf_path, "rb") as f:
        pdf_bytes = f.read()

    # Base64 encode PDF
    base64_encoded_pdf = base64.b64encode(pdf_bytes).decode("utf-8")

    analyze_request = {
        "base64Source": base64_encoded_pdf
    }

    # Start analysis
    poller = await document_intelligence_client.begin_analyze_document(
        "prebuilt-layout",
        analyze_request,
        output_content_format=DocumentContentFormat.MARKDOWN
        
    )
    
    result = await poller.result()
    page_wise_md = result.content.split("<!-- PageBreak -->")

    page_wise_ocr = []

    for page_idx, page in enumerate(result.pages):
        page_wise_context = ""
        for line_idx, line in enumerate(page.lines):
            page_wise_context += line.content + " "

        page_wise_ocr.append(page_wise_context)

    return page_wise_md, page_wise_ocr


In [4]:
page_wise_md, page_wise_ocr = await data_extractor(file_path)

# Text Cleaning:

In [5]:
for i in range(len(page_wise_ocr)):
    print(page_wise_ocr[i])

Product Name: Aripiprazole Tablets USP 5 mg Alembic Touching Lives over 100 years BMR No. & Version No. F1\BMR\00837 & 3.0 Product Code: 30000773 Batch Size in Kg / Liter: 142.500 kg Batch Size in Unit: 1,500,000 Tablets Table of Contents Sr. No. Process Stage Page No. 1 Table of Contents 1 2 Batch Information Sheet 2 3 Abbreviations 3 4 Safety Instruction 4 5 Manufacturing Process 5-55 6 Batch History card 56-57 7 Signature Log 58-59 8 Change History of Document 60 Format No .: C\QASOP\0107-F001-1.0 Effective Date: 09/07/2024 
Product Name: Aripiprazole Tablets USP 5 mg Alembic Touching Lives over 1 100 years BMR No. & Version No. F1\BMR\00837 & 3.0 Product Code: 30000773 Batch Size in Kg / Liter: 142.500 kg Batch Size in Unit: 1,500,000 Tablets BATCH MANUFACTURING RECORD Batch Information Sheet 01 Generic Name of Product Aripiprazole Tablets USP 02 Brand Name of Product NA 03 Label Claim Each tablet contains 5 mg of Aripiprazole USP. 04 Storage Condition Store in tightly closed conta

In [6]:
import re

def remove_header_footer_flexible(text):
    """
    Removes headers/footers without inserting artificial spaces.
    Prevents word merging while preserving original formatting.
    """

    header_patterns = [
        r'Product\s*Name:\s*Aripiprazole\s*Tablets\s*USP\s*5\s*mg.*?Touching\s*Lives\s*over\s*\d+\s*years',
        r'BMR\s*No\.\s*&\s*Version\s*No\.\s*F1\\BMR\\\d+\s*&\s*\d+\.\d+\s*Product\s*Code:\s*\d+',
        r'Batch\s*Size\s*in\s*Kg\s*/\s*Liter:\s*[\d,\.]+\s*kg\s*Batch\s*Size\s*in\s*Unit:\s*[\d,\.]+\s*Tablets',
        r'Product\s*Name:.*?Tablets',
        r'Format\s*No\.?\s*:\s*C\\QA\\SOP\\\d+-F\d+-\d+\.\d+\s*Effective\s*Date\s*:\s*\d{2}/\d{2}/\d{4}',
        r'Alembic\s*Touching\s*Lives\s*over\s*\d+\s*years',
    ]

    footer_patterns = [
        r'Format\s*No\.?\s*:\s*C\\QA\\SOP\\[A-Z0-9\-\.]+\s*Effective\s*Date\s*:\s*\d{2}/\d{2}/\d{4}',
        r'Format\s*No\.?\s*:\s*C\\QASOP\\[A-Z0-9\-\.]+\s*Effective\s*Date\s*:\s*\d{2}/\d{2}/\d{4}',
        r':\s*Format\s*No\.?\s*:\s*C\\QASOP\\[A-Z0-9\-\.]+\s*Effective\s*Date\s*:\s*\d{2}/\d{2}/\d{4}',
        r'No\.?\s*Name\s*Signature.*?Format\s*No\.?\s*:\s*C\\QA\\SOP\\[A-Z0-9\-\.]+\s*Effective\s*Date\s*:\s*\d{2}/\d{2}/\d{4}',
    ]

    cleaned_text = text

    # Remove headers
    for pattern in header_patterns:
        cleaned_text = re.sub(
            pattern,
            '\n',
            cleaned_text,
            flags=re.IGNORECASE | re.DOTALL
        )

    # Remove footers
    for pattern in footer_patterns:
        cleaned_text = re.sub(
            pattern,
            '\n',
            cleaned_text,
            flags=re.IGNORECASE | re.DOTALL
        )

    # Normalize newlines only (DO NOT force spaces)
    cleaned_text = re.sub(r'\n{3,}', '\n\n', cleaned_text)
    cleaned_text = cleaned_text.strip()

    return cleaned_text


In [7]:
cleaned_text_ocr = [
    remove_header_footer_flexible(text)
    for text in page_wise_ocr
]


In [8]:
len(cleaned_text_ocr)

60

In [9]:
cleaned_text_ocr

['Table of Contents Sr. No. Process Stage Page No. 1 Table of Contents 1 2 Batch Information Sheet 2 3 Abbreviations 3 4 Safety Instruction 4 5 Manufacturing Process 5-55 6 Batch History card 56-57 7 Signature Log 58-59 8 Change History of Document 60 Format No .: C\\QASOP\\0107-F001-1.0 Effective Date: 09/07/2024',
 'USP 5 mg Alembic Touching Lives over 1 100 years \n \n BATCH MANUFACTURING RECORD Batch Information Sheet 01 Generic Name of Product Aripiprazole Tablets USP 02 Brand Name of Product NA 03 Label Claim Each tablet contains 5 mg of Aripiprazole USP. 04 Storage Condition Store in tightly closed containers at 25°℃ (77ºF); excursions permitted to 15°-30℃ (59º-86ºF) [see USP Controlled Room Temperature]. 05 Stage /Dosage Form Uncoated Tablet 06 Market/Customer Export /US 07 Reference Document No. MFC/0353-05 08 Manufacturing license No. G/959 09 Manufactured By Alembic Pharmaceuticals Limited, Formulation Unit, Village Panelav, Near Baska, Tal. Halol, Dist. Panchmahal, Gujarat.

In [10]:
cleaned_text = "\n\n".join(cleaned_text_ocr)

In [11]:
print(cleaned_text)

Table of Contents Sr. No. Process Stage Page No. 1 Table of Contents 1 2 Batch Information Sheet 2 3 Abbreviations 3 4 Safety Instruction 4 5 Manufacturing Process 5-55 6 Batch History card 56-57 7 Signature Log 58-59 8 Change History of Document 60 Format No .: C\QASOP\0107-F001-1.0 Effective Date: 09/07/2024

USP 5 mg Alembic Touching Lives over 1 100 years 
 
 BATCH MANUFACTURING RECORD Batch Information Sheet 01 Generic Name of Product Aripiprazole Tablets USP 02 Brand Name of Product NA 03 Label Claim Each tablet contains 5 mg of Aripiprazole USP. 04 Storage Condition Store in tightly closed containers at 25°℃ (77ºF); excursions permitted to 15°-30℃ (59º-86ºF) [see USP Controlled Room Temperature]. 05 Stage /Dosage Form Uncoated Tablet 06 Market/Customer Export /US 07 Reference Document No. MFC/0353-05 08 Manufacturing license No. G/959 09 Manufactured By Alembic Pharmaceuticals Limited, Formulation Unit, Village Panelav, Near Baska, Tal. Halol, Dist. Panchmahal, Gujarat. 10 Custo

In [12]:
import re

def remove_header_footer_flexible(text):
    """
    Removes headers/footers from Batch Manufacturing Record (BMR) documents.
    Preserves original formatting and prevents word merging.
    
    Args:
        text (str): Raw text extracted from PDF
        
    Returns:
        str: Cleaned text with headers/footers removed
    """
    
    # Header patterns - appearing at top of pages
    header_patterns = [
        # Full header block with product name, BMR number, batch size
        r'Product\s*Name:\s*Aripiprazole\s*Tablets\s*USP\s*5\s*mg.*?'
        r'BMR\s*No\.\s*&\s*Version\s*No\.\s*F1\\BMR\\\d+\s*&\s*[\d\.]+\s*Product\s*Code:\s*\d+.*?'
        r'Batch\s*Size\s*in\s*Kg\s*/\s*Liter:\s*[\d,\.]+\s*kg\s*Batch\s*Size\s*in\s*Unit:\s*[\d,\.]+\s*Tablets',
        
        # Standalone product name line
        r'Product\s*Name:\s*Aripiprazole\s*Tablets\s*USP\s*5\s*mg',
        
        # BMR and product code line
        r'BMR\s*No\.\s*&\s*Version\s*No\.\s*F1\\BMR\\\d+\s*&\s*[\d\.]+\s*Product\s*Code:\s*\d+',
        
        # Batch size line
        r'Batch\s*Size\s*in\s*Kg\s*/\s*Liter:\s*[\d,\.]+\s*kg\s*Batch\s*Size\s*in\s*Unit:\s*[\d,\.]+\s*Tablets',
        
        # Company branding
        r'Alembic\s*Touching\s*Lives\s*over\s*\d+\s*years',
        r'Touching\s*Lives\s*over\s*\d+\s*years',
    ]
    
    # Footer patterns - appearing at bottom of pages
    footer_patterns = [
        # **FIXED**: Added \s* before \. to handle space before period in "No ."
        # Format number with effective date (various formats)
        r'Format\s*No\s*\.?\s*:\s*C\\QA\\SOP\\\d+-F\d+-[\d\.]+\s*Effective\s*Date\s*:\s*\d{2}/\d{2}/\d{4}[\'\"]?',
        
        r'Format\s*No\s*\.?\s*:\s*C\\QASOP\\\d+-F\d+-[\d\.]+\s*Effective\s*Date\s*:\s*\d{2}/\d{2}/\d{4}[\'\"]?',
        
        # Footer with preceding colon
        r':\s*Format\s*No\s*\.?\s*:\s*C\\QA\\?SOP\\\d+-F\d+-[\d\.]+\s*Effective\s*Date\s*:\s*\d{2}/\d{2}/\d{4}[\'\"]?',
        
        # Signature table footer combination
        r'Sr\.\s*No\.?\s*Name\s*Signature.*?Format\s*No\s*\.?\s*:\s*C\\QA\\?SOP\\\d+-F\d+-[\d\.]+\s*Effective\s*Date\s*:\s*\d{2}/\d{2}/\d{4}[\'\"]?',
        
        # Generalized version of the previously hardcoded example
        r'Format\s*No\s*\.?\s*:\s*C\\QASOP\\\d{4}-F\d{3}-[\d.]+\s*Effective\s*Date\s*:\s*\d{2}/\d{2}/\d{4}[\'\"]?', 

        # Catch any remaining format line at end of string - more flexible
        r'Format\s*No\s*\.?\s*:\s*C[\\/]+[A-Z]+[\\/]*\d+-F\d+-[\d\.]+\s*Effective\s*Date\s*:\s*\d{2}/\d{2}/\d{4}[\'\"]?',
       
    ]
    
    cleaned_text = text
    
    # Remove headers (preserve line breaks)
    for pattern in header_patterns:
        cleaned_text = re.sub(
            pattern,
            '\n',  # Replace with single newline to maintain separation
            cleaned_text,
            flags=re.IGNORECASE | re.DOTALL
        )
    
    # Remove footers (preserve line breaks)
    for pattern in footer_patterns:
        cleaned_text = re.sub(
            pattern,
            '',  # Replace with empty string
            cleaned_text,
            flags=re.IGNORECASE | re.DOTALL | re.MULTILINE
        )
    
    # Clean up excessive whitespace while preserving structure
    # Remove more than 2 consecutive newlines
    cleaned_text = re.sub(r'\n{3,}', '\n\n', cleaned_text)
    
    # Remove trailing/leading whitespace on each line
    lines = cleaned_text.split('\n')
    lines = [line.rstrip() for line in lines]
    cleaned_text = '\n'.join(lines)
    
    # Remove leading and trailing whitespace from entire text
    cleaned_text = cleaned_text.strip()
    
    return cleaned_text


def extract_and_clean_bmr(pdf_text):
    """
    Complete extraction and cleaning pipeline for BMR documents.
    
    Args:
        pdf_text (str): Raw text extracted from PDF
        
    Returns:
        str: Cleaned and formatted text
    """
    # Apply header/footer removal
    cleaned_text = remove_header_footer_flexible(pdf_text)
    
    # Additional post-processing (optional)
    # Remove page numbers if present
    cleaned_text = re.sub(r'^\d+\s*$', '', cleaned_text, flags=re.MULTILINE)
    
    # Remove standalone horizontal rules or dividers
    cleaned_text = re.sub(r'^[-_=]{3,}\s*$', '', cleaned_text, flags=re.MULTILINE)
    
    # Final cleanup of excessive blank lines
    cleaned_text = re.sub(r'\n{3,}', '\n\n', cleaned_text)
    
    return cleaned_text.strip()


# Example usage
if __name__ == "__main__":
    # Sample text from BMR document
    sample_text = """
    Product Name: Aripiprazole Tablets USP 5 mg
    BMR No. & Version No. F1\\BMR\\00837 & 3.0 Product Code: 30000773
    Batch Size in Kg / Liter: 142.500 kg Batch Size in Unit: 1,500,000 Tablets
    Format No.: C\\QA\\SOP\\0107-F001-1.0 Effective Date: 09/07/2024

    Table of Contents
    Sr. No. Process Stage Page No.
    1 Table of Contents 1
    2 Batch Information Sheet 2 Format No .: C\\QASOP\\0107-F001-1.0 Effective Date: 09/07/2024'
"""
    
    cleaned = extract_and_clean_bmr(sample_text)
    print(cleaned)

Table of Contents
    Sr. No. Process Stage Page No.
    1 Table of Contents 1
    2 Batch Information Sheet 2


In [13]:
cleaned_text_ocr_cl = [
    extract_and_clean_bmr(text)
    for text in page_wise_ocr
]

In [14]:
len(cleaned_text_ocr_cl)

60

In [15]:
cleaned_text_ocr_cl = "\n\n".join(cleaned_text_ocr_cl)

In [16]:
print(cleaned_text_ocr_cl)

Table of Contents Sr. No. Process Stage Page No. 1 Table of Contents 1 2 Batch Information Sheet 2 3 Abbreviations 3 4 Safety Instruction 4 5 Manufacturing Process 5-55 6 Batch History card 56-57 7 Signature Log 58-59 8 Change History of Document 60

BATCH MANUFACTURING RECORD Batch Information Sheet 01 Generic Name of Product Aripiprazole Tablets USP 02 Brand Name of Product NA 03 Label Claim Each tablet contains 5 mg of Aripiprazole USP. 04 Storage Condition Store in tightly closed containers at 25°℃ (77ºF); excursions permitted to 15°-30℃ (59º-86ºF) [see USP Controlled Room Temperature]. 05 Stage /Dosage Form Uncoated Tablet 06 Market/Customer Export /US 07 Reference Document No. MFC/0353-05 08 Manufacturing license No. G/959 09 Manufactured By Alembic Pharmaceuticals Limited, Formulation Unit, Village Panelav, Near Baska, Tal. Halol, Dist. Panchmahal, Gujarat. 10 Customer Batch No. 11 Issued By ( Sign/Date) 12 Received By (Sign/Date) 13 Manufacturing Date 14 Date of Commencement 15

# Chunking

In [17]:
extracted_data = {'document_type': 'BMR ("BATCH MANUFACTURING RECORD") ', 'operation_type': 'Pharmaceutical Manufacturing', 'list_of_process_steps': ['Table of Contents', 'Batch Information Sheet', 'Abbreviations', 'Safety Instruction', 'Manufacturing Process', 'Batch History card', 'Signature Log', 'Change History of Document'], 'product_name': 'Aripiprazole Tablets USP 5 mg', 'product_id': 'F1\\BMR\\00837 & 2.1 / 30000773', 'number_of_operations': '8', 'splitting_text': 'Process Stage'}

In [18]:
list_parent_chunk = extracted_data.get("list_of_process_steps")

In [19]:
list_parent_chunk

['Table of Contents',
 'Batch Information Sheet',
 'Abbreviations',
 'Safety Instruction',
 'Manufacturing Process',
 'Batch History card',
 'Signature Log',
 'Change History of Document']

## Parent wise chunking:

In [25]:
lst = ['Batch Information Sheet',
 'Abbreviations',
 'Safety Instruction',
 'Manufacturing Process',
 'Batch History card',
 'Signature Log',
 'Change History of Document']

In [31]:
import re
from typing import List, Dict

def normalize_text(text: str) -> str:
    text = text.replace("\r", "\n")
    text = re.sub(r'\n+', '\n', text)
    return text.strip()


def build_parent_regex(parent_list: List[str]) -> Dict[str, re.Pattern]:
    patterns = {}
    for p in parent_list:
        escaped = re.escape(p)
        flexible = escaped.replace(r'\ ', r'\s+')
        # must start at line or after newline
        patterns[p] = re.compile(
            rf'(?:(?<=\n)|^)\s*{flexible}\b',
            re.IGNORECASE
        )
    return patterns


def is_toc_like(text_after_header: str) -> bool:
    """
    Detects TOC rows like: 'Batch Information Sheet 2'
    """
    return bool(re.match(r'\s*\d+\s*$', text_after_header))


def split_into_parent_chunks(text: str, parent_list: List[str]) -> List[Dict]:
    text = normalize_text(text)
    patterns = build_parent_regex(parent_list)

    hits = []

    for parent, pattern in patterns.items():
        for m in pattern.finditer(text):
            start = m.start()
            after = text[m.end():m.end() + 20]

            # Skip TOC-style matches
            if is_toc_like(after):
                continue

            hits.append({
                "parent": parent,
                "start": start
            })

    hits = sorted(hits, key=lambda x: x["start"])

    chunks = []
    for i, hit in enumerate(hits):
        s = hit["start"]
        e = hits[i+1]["start"] if i+1 < len(hits) else len(text)

        section = text[s:e].strip()

        chunks.append({
            "parent_id": i + 1,
            "parent_name": hit["parent"],
            "parent_chunk": section
        })

    return chunks


In [32]:
output = split_into_parent_chunks(cleaned_text_ocr_cl, list_parent_chunk)


In [33]:
output

[{'parent_id': 1,
  'parent_name': 'Table of Contents',
  'parent_chunk': 'Table of Contents Sr. No. Process Stage Page No. 1 Table of Contents 1 2 Batch Information Sheet 2 3 Abbreviations 3 4 Safety Instruction 4 5 Manufacturing Process 5-55 6 Batch History card 56-57 7 Signature Log 58-59 8 Change History of Document 60\nBATCH MANUFACTURING RECORD Batch Information Sheet 01 Generic Name of Product Aripiprazole Tablets USP 02 Brand Name of Product NA 03 Label Claim Each tablet contains 5 mg of Aripiprazole USP. 04 Storage Condition Store in tightly closed containers at 25°℃ (77ºF); excursions permitted to 15°-30℃ (59º-86ºF) [see USP Controlled Room Temperature]. 05 Stage /Dosage Form Uncoated Tablet 06 Market/Customer Export /US 07 Reference Document No. MFC/0353-05 08 Manufacturing license No. G/959 09 Manufactured By Alembic Pharmaceuticals Limited, Formulation Unit, Village Panelav, Near Baska, Tal. Halol, Dist. Panchmahal, Gujarat. 10 Customer Batch No. 11 Issued By ( Sign/Date) 